In [2]:
import sys
import os
curPath = os.path.abspath(os.path.dirname('detection2'))
print(curPath)
rootPath = os.path.split(curPath)[0]
sys.path.append(rootPath)
import torch
import torch as t
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchnet import meter
import xarray as xr
import rioxarray as rxr
from torch.nn import functional as F
import math
from models.Conv1d_transformer import  transformer_conv1d,Transformer_Muti_kernel_Conv1d, transformer_mlp,LSTM_conv1d
from models.Conv1d_transformer import Inception_time
from models.STSCDT import Transformer_MKConv1d2
from models.STSCDT import *
from models.Conv1d_transformer import *
from models.LSTM import BiLSTMModel, BiGRUModel
from models.TCN import TCN
from deeplearning.net import *
from load_data import *
from deeplearning.params import *
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

e:\min\detection2


c:\Users\minyu\.conda\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
#---------------------------------Input result-------------------------------------
param_no = 1
region = 'SH'
province =  'SH'
#folder_path = f'G:/Sentinel-SAR/{province}/{region}/blocks/'
folder_path = f'D:/Get_result/{province}/{region}/blocks/'

patch_num = count_tif_files(folder_path)

for i in range(1,patch_num+1):
    
    no = i
    print(i,'/',patch_num, '%')

    PATH = f"D:/Get_result/{province}/{region}/blocks/{region}_R{no}.tif"
    output_path = f'D:/Get_result/{province}/{region}/memory/{region}_R{no}_E{param_no}.tif'
       
    new_net = STSCDT(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6,
                                        ff_h= 256, conv_channels=[128, 64, 32, 16, 1], seq_len = 27, bandDropout=0).to(device) 
    param_path = 'D:\detection2/model_params/STS_CDT_128_L6_ggg.py'
    
    # new_net = Transformer_MKConv1d2(d_model = 32, d_k= 4, heads = 8, dropout=0.5, norm_shape = [27,32], num_encode = 8,
    #                                     ff_h= 64, conv_channels=[32, 64, 32, 16, 1], seq_len = 27).to(device) 
    # param_path = 'D:\detection2/model_params/STS_CDT_32_L8_ggg.py'
    
    # new_net = Transformer(d_model = 128, d_k=16, heads=8, dropout=0.5, norm_shape=[27, 128], ff_h=256, num_encoder=6, mlp = 256, mlp2=128).to(device)
    
    # param_path = 'D:\detection2/model_params/Transformer_128_L6_gg.py'
    
    raster = rxr.open_rasterio(PATH).values
    raster_row, raster_col = (raster.shape)[1], (raster.shape)[2]
    #print( raster_row, raster_col)
    
    row = raster_row
    interval = 10
    cols = [[i, i+interval] for i in range(0, raster_col, interval)]
    if cols[-1][1] != raster_col:
        cols[-1][1] = raster_col

    rows_and_cols = [[0, row]] * len(cols)
    rows_and_cols = [[r, c] for r, c in zip(rows_and_cols, cols)]

    row1, col1 = rows_and_cols[0]
    row2, col2 = rows_and_cols[1]
    is_remove_band = False
    target_band = 'vvhh_exceppt'
    a = []
    for i in range(len(cols)):
        row, col = rows_and_cols[i]
        r = get_result(PATH, row, col, new_net, param_path, device, is_remove_band=is_remove_band, target_band=target_band)
        a.append(r)
    result = np.hstack(a)

    #print('result.shape:',result.shape)
    save_img(output_path, PATH, result)

In [5]:
#--------------------------------- For Single region -------------------------------------
param_no = 1
region = 'YC4'
province =  'JS'
#folder_path = f'G:/Sentinel-SAR/{province}/{region}/blocks/'
folder_path = f'G:/Sentinel-SAR/{province}/{region}/blocks/'

patch_num = count_tif_files(folder_path)

for i in range(1,patch_num+1):
    
    no = i
    print(i,'/',patch_num, '%')

    PATH = f"G:/Sentinel-SAR/{province}/{region}/blocks/{region}_R{no}.tif"
    output_path = f'G:/Sentinel-SAR/{province}/{region}/memory/{region}_R{no}_E{param_no}.tif'
       
    new_net = STSCDT(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6,
                                        ff_h= 256, conv_channels=[128, 64, 32, 16, 1], seq_len = 27, bandDropout=0).to(device) 
    param_path = 'D:\detection2/model_params/STS_CDT_128_L6_ggg.py'
    
#----------------------------------------------------------------------------------------------------------------------------------

    # new_net = Transformer_SKConv1d2(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6,
    #                                      ff_h= 256, conv_channels=[128, 64, 32, 16, 1], seq_len = 27, bandDropout=0).to(device) 
    # param_path = 'D:\detection2/model_params2/SKBP_128_L6_ggg.py'
    
#-----------------------------------------------------------------------------------------------------------------------------------

    # new_net = Transformer_MKDB(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6,
    #                                         ff_h= 256, conv_channels=[128, 64, 32, 16, 1], seq_len = 27).to(device) 
    # param_path = 'D:\detection2/model_params2/MKD_128_L6_ggg.py'
    
#-----------------------------------------------------------------------------------------------------------------------------------

    # new_net = Transformer(d_model = 128, d_k= 16, heads=8, dropout=0.5, norm_shape=[27, 128], ff_h=256, num_encoder=6, mlp = 256, mlp2=128).to(device)
    # param_path = 'D:\detection2/model_params2/Transformer_128_L6_gg.py'

#-----------------------------------------------------------------------------------------------------------------------------------

    # new_net = Transformer_MKNBP(d_model = 128, d_k= 16, heads = 8, dropout=0.5, norm_shape = [27,128], num_encode = 6,
    #                                         ff_h= 256, conv_channels=[128, 64, 32, 16, 1], seq_len = 27).to(device) 
    # param_path = 'D:\detection2/model_params2/MKNBP_128_L6.py'
    
#------------------------------------------------------------------------------------------------------------------------------------

    # new_net = Transformer_MKConv1d2(d_model = 32, d_k= 4, heads = 8, dropout=0.5, norm_shape = [27,32], num_encode = 8,
    #                                     ff_h= 64, conv_channels=[32, 64, 32, 16, 1], seq_len = 27).to(device) 
    # param_path = 'D:\detection2/model_params/STS_CDT_32_L8_ggg.py'
    
    # new_net = Transformer(d_model = 128, d_k=16, heads=8, dropout=0.5, norm_shape=[27, 128], ff_h=256, num_encoder=6, mlp = 256, mlp2=128).to(device)
    
    # param_path = 'D:\detection2/model_params/Transformer_128_L6_gg.py'
    
    raster = rxr.open_rasterio(PATH).values
    raster_row, raster_col = (raster.shape)[1], (raster.shape)[2]
    #print( raster_row, raster_col)
    
    row = raster_row
    interval = 10
    cols = [[i, i+interval] for i in range(0, raster_col, interval)]
    if cols[-1][1] != raster_col:
        cols[-1][1] = raster_col

    rows_and_cols = [[0, row]] * len(cols)
    rows_and_cols = [[r, c] for r, c in zip(rows_and_cols, cols)]

    row1, col1 = rows_and_cols[0]
    row2, col2 = rows_and_cols[1]
    is_remove_band = False
    target_band = 'vvhh_exceppt'
    a = []
    for i in range(len(cols)):
        row, col = rows_and_cols[i]
        r = get_result(PATH, row, col, new_net, param_path, device, is_remove_band=is_remove_band, target_band=target_band)
        a.append(r)
    result = np.hstack(a)

    #print('result.shape:',result.shape)
    save_img(output_path, PATH, result)

4 / 8 %
5 / 8 %
6 / 8 %


e:\min\detection2\load_data.py:336: RuntimeWarning: invalid value encountered in divide
  x_norm = (x_feature - mean)/(std)
e:\min\detection2\load_data.py:336: RuntimeWarning: invalid value encountered in divide
  x_norm = (x_feature - mean)/(std)
e:\min\detection2\load_data.py:336: RuntimeWarning: invalid value encountered in divide
  x_norm = (x_feature - mean)/(std)
e:\min\detection2\load_data.py:336: RuntimeWarning: invalid value encountered in divide
  x_norm = (x_feature - mean)/(std)
e:\min\detection2\load_data.py:336: RuntimeWarning: invalid value encountered in divide
  x_norm = (x_feature - mean)/(std)
e:\min\detection2\load_data.py:336: RuntimeWarning: invalid value encountered in divide
  x_norm = (x_feature - mean)/(std)
e:\min\detection2\load_data.py:336: RuntimeWarning: invalid value encountered in divide
  x_norm = (x_feature - mean)/(std)
e:\min\detection2\load_data.py:336: RuntimeWarning: invalid value encountered in divide
  x_norm = (x_feature - mean)/(std)
e:\min\d

7 / 8 %
8 / 8 %
